In [20]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.",
        metadata={"source": "history_book", "chapter": "Maratha Empire"}
    ),
    Document(
        page_content="Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.",
        metadata={"source": "freedom_book", "chapter": "Indian Independence"}
    ),
    Document(
        page_content="Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.",
        metadata={"source": "science_book", "chapter": "Modern India"}
    ),
]


In [21]:
documents

[Document(metadata={'source': 'history_book', 'chapter': 'Maratha Empire'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
 Document(metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
 Document(metadata={'source': 'science_book', 'chapter': 'Modern India'}, page_content='Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.')]

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv('GROQ_API_KEY')

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model="Llama3-8b-8192")
llm


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001AD6BF32230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AD6BF31720>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings;
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [24]:
from langchain_chroma import Chroma

vectorStore=Chroma.from_documents(documents=documents, embedding=embeddings)

In [25]:
vectorStore

In [26]:
vectorStore.similarity_search("Who is the president of India?")

[Document(id='f6dedbc6-aaef-4429-a535-e829d8b73d9c', metadata={'source': 'science_book', 'chapter': 'Modern India'}, page_content='Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.'),
 Document(id='7037b337-72e7-46c2-9787-07d153a6f83b', metadata={'chapter': 'Modern India', 'source': 'science_book'}, page_content='Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.'),
 Document(id='efcb2f67-4d41-4a53-82de-c9d5832439fa', metadata={'chapter': 'Indian Independence', 'source': 'freedom_book'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
 Document(id='f5989eb9-5ec1-4621-93a8-ccfc45df21c2', metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.')]

In [27]:
##async query
await vectorStore.asimilarity_search("Shivaji")

[Document(id='dc599c22-27ac-40f0-ba7e-cc8da0bc056c', metadata={'source': 'history_book', 'chapter': 'Maratha Empire'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
 Document(id='04475574-fa9c-4bfd-a23b-c7263c4a1dbb', metadata={'chapter': 'Maratha Empire', 'source': 'history_book'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
 Document(id='efcb2f67-4d41-4a53-82de-c9d5832439fa', metadata={'chapter': 'Indian Independence', 'source': 'freedom_book'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
 Document(id='f5989eb9-5ec1-4621-93a8-ccfc45df21c2', metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.')]

In [29]:
vectorStore.similarity_search_with_score("Who is Shivaji?")

[(Document(id='dc599c22-27ac-40f0-ba7e-cc8da0bc056c', metadata={'chapter': 'Maratha Empire', 'source': 'history_book'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
  1.0657446384429932),
 (Document(id='04475574-fa9c-4bfd-a23b-c7263c4a1dbb', metadata={'chapter': 'Maratha Empire', 'source': 'history_book'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
  1.0657446384429932),
 (Document(id='efcb2f67-4d41-4a53-82de-c9d5832439fa', metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
  1.4115279912948608),
 (Document(id='f5989eb9-5ec1-4621-93a8-ccfc45df21c2', metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
  1.4115279912948608)]

In [30]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver=RunnableLambda(vectorStore.similarity_search).bind(k=1)
retriver.batch(["Shivaji", "Kalam"])

[[Document(id='04475574-fa9c-4bfd-a23b-c7263c4a1dbb', metadata={'chapter': 'Maratha Empire', 'source': 'history_book'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.')],
 [Document(id='7037b337-72e7-46c2-9787-07d153a6f83b', metadata={'chapter': 'Modern India', 'source': 'science_book'}, page_content='Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.')]]

In [34]:
vectorStore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriver.batch(["India", "Gandhi"])

[[Document(id='efcb2f67-4d41-4a53-82de-c9d5832439fa', metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.')],
 [Document(id='efcb2f67-4d41-4a53-82de-c9d5832439fa', metadata={'chapter': 'Indian Independence', 'source': 'freedom_book'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.')]]

In [33]:
documents

[Document(metadata={'source': 'history_book', 'chapter': 'Maratha Empire'}, page_content='Chatrapati Shivaji Maharaj was a great Maratha king who built a strong navy.'),
 Document(metadata={'source': 'freedom_book', 'chapter': 'Indian Independence'}, page_content='Mahatma Gandhi led the Indian freedom struggle with non-violence and truth.'),
 Document(metadata={'source': 'science_book', 'chapter': 'Modern India'}, page_content='Dr. A.P.J. Abdul Kalam was known as the Missile Man of India and became the President.')]